###Deltalake & Lakehouse Optimization Usecases

![](/Workspace/Users/infoblisstech@gmail.com/databricks-code-repo/5_all_databricks_workouts/DELTA OPTIMIZATIONS.png)

####1. Handling Data Skew & Query Performance (Optimize & Z-Order)
Scenario: The analytics team reports that queries filtering silver_shipments by source_city and shipment_date are becoming slow as data volume grows.

Task: Run the OPTIMIZE command with ZORDER on the silver_shipments table to co-locate related data in the same files.

Outcome:
Why did we choose source_city and shipment_date for Z-Ordering instead of shipment_id? Think about high cardinality vs. query filtering

In [0]:
%sql
use prodcatalog.logistics;
--describe detail silver_shipments;
--before optimization query, sizeInBytes = 99849

--Run table with optimization
--optimize silver_shipments zorder by (source_city,shipment_date);
--select * from silver_shipments;
describe detail silver_shipments;

--Findings:
--source_city,shipment_date are low cardinality columns, hence zorder by is applied on these columns
--where as shipment_id is high cardinality columns

#### 2. Speeding up Regional Queries (Partition Pruning)
Scenario: The dashboard team reports that queries filtering for orgin_hub_city with "New York" shipments from the gold_core_curated_tbl table are scanning the entire dataset (Terabytes of data), even though New York is only 5% of the data. This is racking up compute costs.

Task: Re-create the gold_core_curated_tbl table partitioned by orgin_hub_city. Run a query filtering for one city to demonstrate "Partition Pruning" (where Spark skips files that don't match the filter).

Outcome: Verify the partition filtering is applied or not, by performing explain plan, check for the PartitionFilters in the output.

In [0]:
%sql
use prodcatalog.logistics;

explain select * from core_curated_tbl where origin_hub_city = 'Newyork';
--Outcome:
--table without partition, PartitionFilters[]

--create or replace table core_curated_tb2 partitioned by (origin_hub_city) as select * from core_curated_tbl;
explain select * from core_curated_tb2 where origin_hub_city = 'Newyork';
--Outcome:
--table with partition, PartitionFilters: [isnotnull(origin_hub_city#16537), (origin_hub_city#16537 = Newyork)]


#### 3. Storage Cost Savings (Vacuum)
Scenario: Your Project pipeline runs every hour, creating many small files and obsolete versions of data. Your storage costs are rising. You need to clean up files that are no longer needed for time travel.

Task: Execute a Vacuum command to remove data files older than the retention threshold.

Outcome: Perform the describe history and find whether vacuum is completed.

In [0]:
%sql
use prodcatalog.logistics;
--vacuum core_curated_tbl retain 168 hours; --7 days data will be retain, rest all removed
describe history core_curated_tbl; --Vacuum start and end with versions
--describe detail core_curated_tb1; --Is throwing error after vacuum operation performed
--select * from core_curated_tbl;

####4. Modern Data Layout (Liquid Clustering)
Scenario: You are redesigning the silver_shipments table. You want to avoid the "small files" problem and need a flexible layout that adapts to changing query patterns automatically without rewriting the table.

Task: Re-create the silver_shipments table using Liquid Clustering on the shipment_id column.

Outcome: Liquid Clustering over traditional partitioning when the cardinality of shipment_id is very high.

In [0]:
%sql
--select * from silver_shipments;

create table if not exists silver_shipments_cluster
as 
select * from silver_shipments
cluster by (shipment_id);

describe history silver_shipments_cluster; 
--by default shipment_id order by asc after clustering using cluster by
--select * from silver_shipments_cluster order by shipment_id desc;

describe detail silver_shipments_cluster; 

#### 5. Cost Efficient Environment Cloning (Shallow Clone)
Scenario: The QA team needs to test an update on the gold_core_curated_tbl table. The table is 5TB in size. You cannot afford to duplicate the storage cost just for a test and the update should not affect the copied table.

Task: Create a Shallow Clone of the gold table for the QA team.

Outcome: If we delete records from the source table (gold_core_curated_tbl), will the QA table (gold_core_curated_tbl_qa) be affected? Why or why not?

In [0]:
%sql
--select * from core_curated_tbl;
--describe detail core_curated_tbl;


create table if not exists core_curated_tbl_sclone
shallow clone core_curated_tbl;

describe history core_curated_tbl_sclone;


--insert new record in core_curated_tbl and check its not reflected in shallow cloned table
--insert into core_curated_tbl values (1234567,'vivekbharathi','data engineer','bangalore',12345.33,2026,2,'Bangalore-Chennai',23.12,4610.87,current_timestamp);

--select count(1) from core_curated_tbl_sclone; 
-- (newly inserted record is not reflected in shallow cloned table even it clone the same data file, hence transaction logs are not shared)

#### 6. Disaster Recovery (Time Travel & Restore)
Scenario: A junior data engineer accidentally ran a logic error that corrupted the gold_core_curated_tbl table 15 minutes ago. You need to revert the table to its previous state immediately.

Task: Use Delta Lake's Restore feature to roll back the table.

Outcome:What is the difference between querying with VERSION AS OF (Time Travel) and running RESTORE?

In [0]:
%sql


restore table core_curated_tbl
to version as of 2;

describe history core_curated_tbl;
/* 
The new inserted record in version 3, later table restored to version 2. 
The insert row changes reverted back to older version.
*/
